# QLoRA training — SCOTUS classifier (Kaggle)

Run this notebook on Kaggle:
1. Notebook settings → **Accelerator**: **GPU T4 x2** — not P100. `bitsandbytes` 4-bit
   quantization requires CUDA compute capability ≥7.0 (Turing+); the P100 is Pascal
   (sm_60) and fails with a CUDA kernel symbol error on 4-bit ops. If pushing via the
   `kaggle kernels push` API rather than the UI, pass `--accelerator NvidiaTeslaT4`
   explicitly — `enable_gpu: true` in kernel-metadata.json alone can still assign a P100.
   **Internet**: On (needed to pull the base model/dataset from Hugging Face).
2. The first code cell clones [Indiandude123/domain_adapted_slm](https://github.com/Indiandude123/domain_adapted_slm)
   automatically — no manual setup needed.
3. Set `HF_TOKEN` as a Kaggle secret if you plan to `--push-to-hub` adapter checkpoints
   (recommended — Kaggle's interactive session state isn't durable across restarts, so
   pushing checkpoints as training progresses is the main mitigation for hitting the
   session time limit mid-run).
4. Run cells top to bottom. The unweighted and weighted runs are independent — run them in
   separate sessions if you're tight on the 30 GPU-hrs/week quota.

In [ ]:
import os

# Restrict to a single GPU. The T4 x2 instance exposes 2 GPUs, and device_map="auto" (used
# when loading the quantized model) will shard a model across both — but Phi-3-mini in 4-bit
# is only ~2.5GB and fits comfortably on one T4. Combining that auto-sharding with the HF
# Trainer's own multi-GPU handling caused a backward-pass OOM on GPU 1 at ~10.6/14.56GB used
# (confirmed by a failed run), not a real out-of-memory condition. Single-GPU avoids it.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if not os.path.exists("repo"):
    !git clone -q https://github.com/Indiandude123/domain_adapted_slm.git repo
os.chdir("repo")

!pip install -q -r requirements.txt

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret found — push-to-hub checkpointing will be unavailable.")

from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])

## Sanity check on a tiny subset

Highest-risk step (Day 2 of the plan): confirm the QLoRA + SEQ_CLS loop runs end-to-end
without OOM before committing GPU-hours to a full run.

In [ ]:
from src.data.load import load_scotus
from src.data.preprocess import tokenize_batch
from src.model.qlora import apply_lora, load_quantized_classifier
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

dataset = load_scotus()
small_train = dataset["train"].select(range(32))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = load_quantized_classifier(MODEL_NAME, num_labels=14)
model = apply_lora(model)
model.print_trainable_parameters()

sample = tokenize_batch(small_train[:4], tokenizer, max_length=512)
print({k: len(v) for k, v in sample.items()})

## Full training runs

Unweighted baseline, then the class-weighted-loss run — the comparison between the two is
the evidence behind the imbalance-handling resume bullet.

In [ ]:
import subprocess

subprocess.run(
    ["python", "-m", "src.train.run_train", "--config", "configs/scotus_phi3.yaml"], check=True
)

In [ ]:
subprocess.run(
    ["python", "-m", "src.train.run_train", "--config", "configs/scotus_phi3.yaml", "--weighted"],
    check=True,
)

Next: run `src/eval/metrics.py` against the held-out test split for both checkpoints and
`src/eval/benchmark.py` for latency/memory numbers (Day 5 of the plan).